# Config B - Continuous-State ICU-Sepsis (Deep RL)

This notebook **only calls** the implementation in [`sepsis_rl.py`](sepsis_rl.py).
All the agent / training / evaluation / plotting logic lives there; here we just
run it and look at the results. Every figure is saved to the **`outputs/`** directory.

**Action space is `Discrete(25)`** (5 vasopressor x 5 IV-fluid levels) - the same as
Config A. What is continuous in Config B is the **observation** (47-dim clinical
feature vector). We therefore use the discrete-action algorithms **DQN, PPO, A2C**.

| Run | Normalization | Purpose |
|-----|---------------|---------|
| DQN, PPO (v1) | none | baseline on raw observations |
| DQN-v2, PPO-v2, A2C | VecNormalize | clean ablation: effect of obs normalization |

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import sepsis_rl as srl

srl.ensure_dirs(srl.OUTPUT_DIR, srl.MODELS_DIR, srl.LOGS_DIR)
print('Device           :', srl.DEVICE)
print('Plots saved to   :', srl.OUTPUT_DIR + '/')
print('Algorithms       :', list(srl.ALGOS))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Device           : cuda
Plots saved to   : outputs/
Algorithms       : ['DQN', 'PPO', 'A2C']


## 1. Environment & random baseline

The full clinical environment (`make_clinical_env`) stacks three failure-mode
wrappers on top of the continuous base env: episodic observation noise, episodic
missing labs, and rare acute death events. The random policy is the target to beat.

In [2]:
from envs.wrappers import make_clinical_env

env = make_clinical_env()
print('Observation space:', env.observation_space)
print('Action space     :', env.action_space)
env.close()

# Random baseline (use 1000 episodes for the report)
random_stats = srl.random_baseline(n_episodes=1000)
RANDOM_RETURN = random_stats['return']

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02
Observation space: Box(-inf, inf, (47,), float32)
Action space     : Discrete(25)


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


[random] return=0.5800  survival=67.4%


## 2. Training

`TIMESTEPS` controls the budget. Default is **fast** (a few minutes per agent).
For final, report-quality results set it to `1_000_000` (several hours total).

In [3]:
TIMESTEPS = 150_000      # <- para resultados finais usa 1_000_000
EVAL_FREQ = 5_000
N_EVAL    = 200           # aumentado de 50 para curvas mais suaves

# --- v1: raw observations (no normalization) ---
_, dqn_tag = srl.train_agent('DQN', timesteps=TIMESTEPS, normalize=False,
                             eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL)
_, ppo_tag = srl.train_agent('PPO', timesteps=TIMESTEPS, normalize=False,
                             eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL)

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


Output()

[dqn] training complete (150,000 steps).


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


Output()

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


[ppo] training complete (150,000 steps).


### v2: with `VecNormalize`

The 47 features live on wildly different scales, so normalizing observations is the
single biggest lever for these agents. v2 changes **only** the normalization vs v1.

In [4]:
_, dqn2_tag = srl.train_agent('DQN', timesteps=TIMESTEPS, normalize=True,
                             eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL)
_, ppo2_tag = srl.train_agent('PPO', timesteps=TIMESTEPS, normalize=True,
                             eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL)
_, a2c_tag  = srl.train_agent('A2C', timesteps=TIMESTEPS, normalize=True,
                             eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL)

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


Output()

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


[dqn_v2] training complete (150,000 steps).


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


Output()

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


## 3. Learning curves

Read from each run's `evaluations.npz`, EMA-smoothed (faint line = raw eval returns).

In [5]:
# Curva completa (todos os passos)
srl.plot_learning_curves(
    {dqn_tag: 'DQN (v1)', ppo_tag: 'PPO (v1)',
     dqn2_tag: 'DQN-v2', ppo2_tag: 'PPO-v2', a2c_tag: 'A2C'},
    baseline=RANDOM_RETURN,
)

# Zoom nos primeiros 200k passos - onde a aprendizagem acontece
srl.plot_learning_curves(
    {dqn_tag: 'DQN (v1)', ppo_tag: 'PPO (v1)',
     dqn2_tag: 'DQN-v2', ppo2_tag: 'PPO-v2', a2c_tag: 'A2C'},
    baseline=RANDOM_RETURN,
    zoom_steps=200_000,
    filename='configB_learning_curves_zoom.png',
)

Saved outputs\configB_learning_curves.png
Saved outputs\configB_learning_curves_zoom.png


'outputs\\configB_learning_curves_zoom.png'

## 4. Robustness evaluation

Each agent is evaluated on its own env per failure mode (Clean / Noisy / Missing /
Acute / All). Survival is read from the terminal reward (death -> 0, survival -> ~1).

In [6]:
N_EVAL_EPISODES = 1000     # 1000 for the report

results = {
    'DQN (v1)': srl.evaluate_conditions(dqn_tag,  'DQN', n_episodes=N_EVAL_EPISODES),
    'PPO (v1)': srl.evaluate_conditions(ppo_tag,  'PPO', n_episodes=N_EVAL_EPISODES),
    'DQN-v2':   srl.evaluate_conditions(dqn2_tag, 'DQN', n_episodes=N_EVAL_EPISODES),
    'PPO-v2':   srl.evaluate_conditions(ppo2_tag, 'PPO', n_episodes=N_EVAL_EPISODES),
    'A2C':      srl.evaluate_conditions(a2c_tag,  'A2C', n_episodes=N_EVAL_EPISODES),
}

srl.results_table(results, random_ret=RANDOM_RETURN, condition='All')

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


[dqn] All      n=1000  return=0.5715  survival=64.7%  intensity=0.404
[dqn] Clean    n= 673  return=0.6217  survival=69.7%  intensity=0.406
[dqn] Noisy    n= 148  return=0.6093  survival=68.2%  intensity=0.388
[dqn] Missing  n= 140  return=0.5841  survival=66.4%  intensity=0.419
[dqn] Acute    n=  92  return=-0.0720  survival=0.0%  intensity=0.399


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


[ppo] All      n=1000  return=0.5633  survival=65.1%  intensity=0.488
[ppo] Clean    n= 678  return=0.6253  survival=71.2%  intensity=0.482
[ppo] Noisy    n= 137  return=0.5228  survival=60.6%  intensity=0.496
[ppo] Missing  n= 159  return=0.5201  survival=61.0%  intensity=0.508
[ppo] Acute    n=  71  return=-0.0949  survival=0.0%  intensity=0.510


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


...

[ppo_v2] All      n=1000  return=0.5808  survival=66.0%  intensity=0.453
[ppo_v2] Clean    n= 675  return=0.6307  survival=71.1%  intensity=0.451
[ppo_v2] Noisy    n= 149  return=0.5780  survival=65.8%  intensity=0.455
[ppo_v2] Missing  n= 150  return=0.6041  survival=68.0%  intensity=0.442
[ppo_v2] Acute    n=  87  return=-0.0688  survival=0.0%  intensity=0.482


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


[a2c_v2] All      n=1000  return=0.5871  survival=66.1%  intensity=0.404
[a2c_v2] Clean    n= 656  return=0.6372  survival=71.3%  intensity=0.405
[a2c_v2] Noisy    n= 146  return=0.6094  survival=67.8%  intensity=0.392
[a2c_v2] Missing  n= 168  return=0.6099  survival=67.9%  intensity=0.419
[a2c_v2] Acute    n=  78  return=-0.0625  survival=0.0%  intensity=0.404


,Agent,Return (All),Survival (All),Intensity (All),vs Random
0,DQN (v1),0.5715,64.7%,0.404,-1.5%
1,PPO (v1),0.5633,65.1%,0.488,-2.9%
2,DQN-v2,0.5753,65.2%,0.377,-0.8%
3,PPO-v2,0.5808,66.0%,0.453,+0.1%
4,A2C,0.5871,66.1%,0.404,+1.2%


In [7]:
srl.plot_robustness(results, metric='return',   baseline=RANDOM_RETURN,
                    filename='configB_robustness_return.png')
srl.plot_robustness(results, metric='survival', baseline=random_stats['survival'],
                    filename='configB_robustness_survival.png')

Saved outputs\configB_robustness_return.png


Saved outputs\configB_robustness_survival.png


'outputs\\configB_robustness_survival.png'

## 5. Optional hyperparameter tuning (Optuna)

This section is intentionally disabled by default so the notebook can run end-to-end in a reasonable time. Set `RUN_OPTUNA=True` to run the short proxy search, and set `RUN_TUNED=True` only after Optuna if you want full-budget tuned training.


In [8]:
RUN_OPTUNA = False  # Set True only when you want the extra hyperparameter-search run.
studies = {}

if RUN_OPTUNA:
    for algo in ["DQN", "PPO", "A2C"]:
        studies[algo] = srl.tune(algo, n_trials=15, timesteps=50_000, normalize=True)
        srl.save_optuna_plots(studies[algo], name=algo)

    for algo, study in studies.items():
        print(f"{algo}: best return={study.best_value:.4f}  params={study.best_params}")

    import json as _json
    with open("optuna_best_params.json", "w") as f:
        _json.dump({algo: study.best_params for algo, study in studies.items()}, f, indent=2)
    print("Best params saved to optuna_best_params.json")
else:
    print("Optuna skipped. Set RUN_OPTUNA=True for the optional hyperparameter search.")


Optuna skipped. Set RUN_OPTUNA=True for the optional hyperparameter search.


In [9]:
if RUN_OPTUNA:
    srl.plot_best_trial_curves(
        ['DQN', 'PPO', 'A2C'],
        baseline=RANDOM_RETURN,
    )
else:
    print("Best-trial curve skipped because RUN_OPTUNA=False.")


Best-trial curve skipped because RUN_OPTUNA=False.


### Optional full-budget retraining

Plug Optuna's best hyperparameters into a full-budget run. This is disabled by default because it can take several hours.


In [10]:
RUN_TUNED = False  # Set True after RUN_OPTUNA=True if you want full-budget tuned training.
TIMESTEPS_TUNED = 1_000_000

tuned_tags = {}

if RUN_TUNED:
    if not studies:
        raise RuntimeError("Run Optuna first: set RUN_OPTUNA=True and execute the tuning cell.")

    for best_algo in ['DQN', 'PPO', 'A2C']:
        best_hp = dict(studies[best_algo].best_params)
        _, tag = srl.train_agent(
            best_algo, timesteps=TIMESTEPS_TUNED, normalize=True,
            hyperparams=best_hp, tag=f'{best_algo.lower()}_tuned',
            eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL,
        )
        tuned_tags[best_algo] = tag

    srl.plot_learning_curves(
        {tuned_tags["DQN"]: "DQN (tuned)",
         tuned_tags["PPO"]: "PPO (tuned)",
         tuned_tags["A2C"]: "A2C (tuned)"},
        baseline=RANDOM_RETURN,
        zoom_steps=TIMESTEPS_TUNED,
        filename="configB_all_tuned_curves.png",
    )

    srl.plot_learning_curves(
        {tuned_tags["DQN"]: "DQN (tuned)",
         tuned_tags["PPO"]: "PPO (tuned)",
         tuned_tags["A2C"]: "A2C (tuned)"},
        baseline=RANDOM_RETURN,
        zoom_steps=300_000,
        filename="configB_all_tuned_curves_zoom.png",
    )

    results_tuned = {
        f"{algo} (tuned)": srl.evaluate_conditions(tag, algo, n_episodes=N_EVAL_EPISODES)
        for algo, tag in tuned_tags.items()
    }
    display(srl.results_table(results_tuned, random_ret=RANDOM_RETURN))
    srl.plot_robustness(results_tuned, metric="return", baseline=RANDOM_RETURN,
                        filename="configB_tuned_robustness.png")
else:
    print("Full-budget tuned training skipped. Set RUN_TUNED=True after running Optuna.")


Full-budget tuned training skipped. Set RUN_TUNED=True after running Optuna.


In [11]:
if tuned_tags:
    for algo, tag in tuned_tags.items():
        srl.plot_learning_curves(
            {tag: f"{algo} (tuned)"},
            baseline=RANDOM_RETURN,
            show_trend=True,
            filename=f"configB_{algo.lower()}_tuned_trend.png",
        )
else:
    print("Tuned trend plots skipped because RUN_TUNED=False.")


Tuned trend plots skipped because RUN_TUNED=False.


In [12]:
if tuned_tags:
    srl.plot_curves_grid(
        {'dqn_tuned': 'DQN (tuned)',
         'ppo_tuned': 'PPO (tuned)',
         'a2c_tuned': 'A2C (tuned)'},
        baseline=RANDOM_RETURN,
        filename='configB_tuned_grid.png',
    )
else:
    print("Tuned grid plot skipped because RUN_TUNED=False.")


Tuned grid plot skipped because RUN_TUNED=False.


## 6. Creative extension - robustness + clinical interpretability

Our creative extension audits the trained agents along two clinical axes:

1. **Robustness** under the three failure modes (Clean / Noisy / Missing / Acute) -
   already computed above via `evaluate_conditions` and `plot_robustness`.
2. **Interpretability**: (a) the *treatment-intensity* and *dose grid* each policy
   prescribes, and (b) *feature importance* of the DQN Q-network - which of the 47
   clinical variables actually drive its decisions.

This connects the learned policies back to clinical meaning (the whole point of the
`lam` parsimony penalty and the sepsis-severity features) and does so **without**
altering the required environment.

### 6.1 Treatment intensity & dose grid

Lower intensity at equal survival is clinically preferable. The 5x5 grid shows what each policy actually prescribes (vasopressor x IV fluid).

In [13]:
# Mean treatment intensity per failure mode (already collected during evaluation)
srl.plot_robustness(results, metric='intensity', baseline=None,
                    filename='configB_intensity.png')

# 5x5 dose grid (vasopressor x IV fluid) for every agent
srl.plot_dose_grid(results, filename='configB_dose_grid.png')

Saved outputs\configB_intensity.png


Saved outputs\configB_dose_grid.png


'outputs\\configB_dose_grid.png'

### 6.2 DQN feature importance (Q-value perturbation)

Permute each feature and measure the mean |Delta Q| on the chosen action. Red bars = established sepsis-severity markers; a credible agent should rely on them.

In [14]:
# Use the normalized DQN (dqn2_tag); importance is computed on the same
# normalized inputs the network saw during training.
importances, order = srl.feature_importance_dqn(dqn2_tag, n_states=500, top_n=20)

make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02


make_sepsis_env | lam=0.02 → intensity penalty active
make_sepsis_env | sofa_bias=5.0 → mean start SOFA: 9.02
Saved outputs\configB_feature_importance.png
Top 10 decision-relevant features:
   1. HR               0.09157 <- sepsis marker
   2. Milrinone        0.07819
   3. Weight           0.07362
   4. Age              0.06985
   5. AST              0.06443
   6. Insulin          0.06188
   7. Norepinephrine   0.05777
   8. WBC              0.05572
   9. Phenylephrine    0.05512
  10. Chloride         0.05479


### 6.3 Honest Config A vs Config B comparison

Config A numbers are read from `configA_results.json` (produced by the Config A notebook) - never hard-coded. **Run the Config A notebook first.**

In [15]:
# Cross-config table using REAL Config A metrics from configA_results.json
df_compare = srl.compare_configs(results, configA_path='configA_results.json',
                                 condition='All')
df_compare

,Config,Agent,Return,Survival,Intensity
0,A,Random,0.5877,68.7%,0.494
1,A,Policy Iteration,0.7507,78.8%,0.163
2,A,Q-Learning,0.6115,71.3%,0.486
3,A,SARSA,0.6313,72.6%,0.462
4,B,DQN (v1),0.5715,64.7%,0.404
5,B,PPO (v1),0.5633,65.1%,0.488
6,B,DQN-v2,0.5753,65.2%,0.377
7,B,PPO-v2,0.5808,66.0%,0.453
8,B,A2C,0.5871,66.1%,0.404
